# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [4]:
df['revenue'] = df['price']*df['qty']

print(f"Total revenue: {sum(df['revenue'])}, total units: {sum(df['qty'])}")

Total revenue: 8520.0, total units: 783


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [18]:
q2 = df.groupby('category', as_index=False)['revenue'].sum().sort_values('revenue', ascending=False)
q2['share'] = q2['revenue'] / q2['revenue'].sum()
q2

,category,revenue,share
1,Food,4293.0,0.503873
2,Merch,1771.5,0.207923
0,Drink,1554.0,0.182394
3,RainGear,901.5,0.105810


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [17]:
q3 = df.groupby('vendor_id', as_index=False)['revenue'].agg(avg='mean').sort_values('avg', ascending=False)
q3

,vendor_id,avg
0,V-01,22.595745
3,V-18,21.750000
1,V-05,20.580645
2,V-10,20.314286


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [30]:
print(f"Merch revenue share: {(q2.loc[q2["category"] == "Merch", "share"].iloc[0] * 100).round(1)}%")

Merch revenue share: 20.8%


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [36]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

n_before = len(df)
rev_before = df['revenue'].sum()

merged = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one', indicator=True)

# unmatched vendor
print("Vendor ids not in lookup:",merged.loc[merged['_merge'] == 'left_only', 'vendor_id'].unique())

# label rows
merged['vendor_name'] = merged['vendor_name'].fillna('Unknown (' + merged['vendor_id'] + ')')
merged = merged.drop(columns='_merge')

# assertions
assert len(merged) == n_before, "Row count changed"
assert np.isclose(merged['revenue'].sum(), rev_before), "Revenue total changed"
print(f"Rows: {n_before} -> {len(merged)}, Revenue: {rev_before:,.2f} -> {merged['revenue'].sum():,.2f}")

Vendor ids not in lookup: ['V-18']
Rows: 400 -> 400, Revenue: 8,520.00 -> 8,520.00


**The unmatched vendor, and what I did about it:** The unmatched vendor was V-18. I decided to keep its rows and label them Unknown (V-18), as dropping them would make the revenue check fail.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [41]:
pivot = merged.pivot_table(
    index='vendor_name',
    columns='category',
    values='revenue',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total',
)

row_order = pivot.drop('Total').sort_values('Total', ascending=False).index.tolist() + ['Total']
col_order = pivot.drop(columns='Total').loc['Total'].sort_values(ascending=False).index.tolist() + ['Total']
pivot = pivot.loc[row_order, col_order]
pivot

category,Food,Merch,Drink,RainGear,Total
vendor_name,,,,,
Unknown (V-18),1018.5,508.5,582.0,240.0,2349.0
Cav Merch North,1054.5,400.5,502.5,175.5,2133.0
Hoos Burgers,1338.0,373.5,171.0,241.5,2124.0
Rotunda Tacos,882.0,489.0,298.5,244.5,1914.0
Total,4293.0,1771.5,1554.0,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [45]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(q2['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(merged) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a) Food is ~50.4% of the 8,520.00 total revenue,  but average order is 23.08 against Merch's 22.42, so they are pretty much the same for what a single
sale is worth. I would tell my vendors to prioritize drinks, as V-01 gained 171.00 in Drink revenue across 12 orders, while V-18 only gained 582.00 across 31, which is more than triple the revenue from a comparable stand, and V-01 is the highest grossing Food counter in the report at 1,338.00 revenue. V-01 is selling food to a line of people without a drink. Next game, V-01 should
prompt a drink on every food order and price a combo, like a 4.50 drink attached to even half of its 55 Food orders would add roughly 120 at close to zero extra labor. Merch is also worth considering too, as it's the number-two category at 1,771.50 (20.8%) on high-value items.


b) Q6, the pivot table, is the least trustworthy. It splits 400 orders into 16 vendor × category cells with about 25 orders each. All of RainGear is only 46 orders, so each vendor's RainGear cell is only on 9 to 15 sales. With unit prices ranging from 4.50 to 24.00 and quantities of 1 to 3, one 24.00 × 3 order moves a cell by 72 on a base of about 200. So V-05's RainGear (244.50) beating V-10's (175.50) is just noise. The category totals in Q2 are fine, but the per-cell breakdown should not be read as a ranking.